# Laboratoire 6 — Quantum Support Vector Machine (QSVM)

**Quantum Machine Learning — Master/PhD**

**Objectifs :** Implémenter un noyau (kernel) quantique et l'utiliser dans un SVM pour la classification.

- **Partie A :** Fidelity quantum kernel avec PennyLane (`qml.kernels.embedding_kernel_matrix`)
- **Partie B :** Pipeline QSVM : encodage → kernel matrix → SVM classique (sklearn `SVC`)
- **Partie C :** Quantum kernel alignment : optimisation des paramètres d'encoding
- **Partie D :** Implémentation Qiskit ML : `FidelityQuantumKernel`, `QSVC`
- **Partie E :** Comparaison SVM classique (RBF, poly) vs. QSVM sur Iris / Wine

**Références :** HavlÍček et al. (2019), *Nature* ; [SP21] Ch. 6 ; Schuld (2024) *Supervised Learning with Quantum Computers*

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_iris, load_wine
import warnings
warnings.filterwarnings('ignore')

print(f"PennyLane : {qml.__version__}")
print(f"scikit-learn : {__import__('sklearn').__version__}")

# Optionnel : Qiskit Machine Learning
try:
    from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    from qiskit_machine_learning.algorithms import QSVC
    from qiskit_aer import AerSimulator
    QISKIT_ML_AVAILABLE = True
    print("Qiskit Machine Learning : disponible")
except ImportError:
    QISKIT_ML_AVAILABLE = False
    print("Qiskit Machine Learning : non installé (pip install qiskit-machine-learning)")

---
## Partie A : Fidelity quantum kernel avec PennyLane

Le *fidelity quantum kernel* est défini par :

$$k(x_i, x_j) = |\langle 0 | U^\dagger(x_j) U(x_i) | 0 \rangle|^2$$

où $U(x)$ est un circuit d'encodage (feature map). Cette quantité mesure la similarité (fidélité) entre deux états encodés.

In [ ]:
def create_fidelity_kernel(n_qubits, n_layers=1):
    """Crée un fidelity quantum kernel avec PennyLane.
    
    Args:
        n_qubits: Nombre de qubits (doit correspondre à la dimension des données)
        n_layers: Nombre de couches variationnelles
    
    Retourne:
        kernel_fn: fonction kernel(x1, x2) -> float
    """
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_circuit(x1, x2):
        """Circuit pour le fidelity kernel : |<0|U†(x2)U(x1)|0>|²."""
        qml.AngleEmbedding(x1, wires=range(n_qubits))
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits))
        return qml.probs(wires=range(n_qubits))

    def kernel_fn(x1, x2):
        """Kernel : probabilité de l'état |0...0⟩ après le circuit."""
        return kernel_circuit(x1, x2)[0]

    return kernel_fn


# Test sur des données aléatoires
n_qubits = 4
kernel_fn = create_fidelity_kernel(n_qubits)

x_test1 = np.random.randn(n_qubits)
x_test2 = np.random.randn(n_qubits)

k_value = kernel_fn(x_test1, x_test2)
print(f"Kernel test : k(x₁, x₁) = {kernel_fn(x_test1, x_test1):.4f}  (devrait être 1)")
print(f"Kernel test : k(x₁, x₂) = {k_value:.4f}  (similarité entre deux points aléatoires)")
print(f"Kernel test : k(x₁, -x₁) = {kernel_fn(x_test1, -x_test1):.4f}  (points opposés)")

---
## Partie B : Pipeline QSVM complet

Le pipeline QSVM se compose de :
1. **Encodage** : les données classiques sont encodées en états quantiques
2. **Matrice de kernel** : on calcule $K_{ij} = k(x_i, x_j)$ pour toutes les paires d'échantillons
3. **SVM classique** : on utilise `sklearn.svm.SVC` avec `kernel='precomputed'`

In [ ]:
def build_kernel_matrix(kernel_fn, X1, X2):
    """Construit une matrice de kernel entre deux ensembles de points."""
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            K[i, j] = kernel_fn(X1[i], X2[j])
    return K


def qsvm_pipeline(X_train, y_train, X_test, y_test, kernel_fn):
    """Pipeline QSVM complet.
    
    Étapes :
    1. Calcul de la matrice de kernel pour l'entraînement
    2. Entraînement du SVC avec kernel pré-calculé
    3. Calcul de la matrice de kernel pour le test
    4. Prédiction et évaluation
    """
    print("Calcul de la matrice de kernel (entraînement)...")
    K_train = build_kernel_matrix(kernel_fn, X_train, X_train)

    print("Entraînement du SVM avec kernel pré-calculé...")
    svm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_train)

    print("Calcul de la matrice de kernel (test)...")
    K_test = build_kernel_matrix(kernel_fn, X_test, X_train)

    y_pred = svm.predict(K_test)
    acc = accuracy_score(y_test, y_pred)
    return svm, y_pred, acc


# Chargement des données
iris = load_iris()
X, y = iris.data[:, :4], iris.target  # 4 features → 4 qubits

# Binariser pour une classification à 2 classes (plus simple)
mask = y != 2
X, y = X[mask], y[mask]

# Normalisation dans [0, π] pour l'angle encoding
scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Données : train={len(X_train)}, test={len(X_test)}, features={X.shape[1]}")

# Pipeline QSVM
kernel_fn = create_fidelity_kernel(n_qubits=4, n_layers=1)
svm_q, y_pred_q, acc_q = qsvm_pipeline(X_train, y_train, X_test, y_test, kernel_fn)

print(f"\n=== Résultats QSVM ===")
print(f"Accuracy : {acc_q:.4f}")
print(classification_report(y_test, y_pred_q, target_names=['setosa', 'versicolor']))

---
## Partie C : Quantum Kernel Alignment

Le *quantum kernel alignment* (QKA) consiste à optimiser les paramètres de la feature map pour maximiser l'alignement du kernel avec les données.

$$A(K, yy^T) = \frac{\langle K, yy^T \rangle_F}{\sqrt{\langle K, K \rangle_F \langle yy^T, yy^T \rangle_F}}$$

où $\langle \cdot, \cdot \rangle_F$ est le produit de Frobenius et $y$ les labels.

In [ ]:
def kernel_alignment(K, y):
    """Calcule l'alignement du kernel avec les labels.
    
    A(K, yy^T) = <K, yy^T>_F / sqrt(<K,K>_F * <yy^T,yy^T>_F)
    """
    y = y.reshape(-1, 1)
    Y = y @ y.T  # yy^T : +1 si même classe, -1 sinon
    K_centered = K - np.mean(K)
    Y_centered = Y - np.mean(Y)
    num = np.sum(K_centered * Y_centered)
    den = np.sqrt(np.sum(K_centered**2) * np.sum(Y_centered**2))
    return num / den if den > 0 else 0.0


def create_parameterized_kernel(n_qubits, n_layers=1):
    """Crée un kernel avec paramètres de encoding optimisables."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_circuit(x1, x2, params):
        # Encoding pondéré par les paramètres
        qml.AngleEmbedding(params[:n_qubits] * x1, wires=range(n_qubits))
        qml.AngleEmbedding(params[n_qubits:] * x2, wires=range(n_qubits))
        qml.adjoint(qml.AngleEmbedding)(params[:n_qubits] * x2, wires=range(n_qubits))
        qml.adjoint(qml.AngleEmbedding)(params[n_qubits:] * x1, wires=range(n_qubits))
        return qml.probs(wires=0)  # On mesure seulement le premier qubit

    # Version plus simple : encoding standard
    @qml.qnode(dev)
    def kernel_circuit_simple(x1, x2, params):
        qml.AngleEmbedding(x1 * params[0], wires=range(n_qubits))
        qml.adjoint(qml.AngleEmbedding)(x2 * params[0], wires=range(n_qubits))
        return qml.probs(wires=range(n_qubits))

    def kernel_fn(x1, x2, params):
        return kernel_circuit_simple(x1, x2, params)[0]

    return kernel_fn


def compute_kernel_matrix(kernel_fn, X1, X2, params):
    """Matrice de kernel avec paramètres."""
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            K[i, j] = kernel_fn(X1[i], X2[j], params)
    return K

In [ ]:
from scipy.optimize import minimize

# Préparer les données (Iris, classes 0 vs 1)
X_bin, y_bin = iris.data[:100], iris.target[:100]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_bin = scaler.fit_transform(X_bin)
y_bin = np.where(y_bin == 0, -1, 1)  # {-1, +1} pour l'alignement

X_train, X_test, y_train, y_test = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

kernel_p = create_parameterized_kernel(n_qubits=4)

def objective(scale_param):
    """Objectif : -alignment (à minimiser)."""
    params = np.array([scale_param] * 4)
    K = compute_kernel_matrix(kernel_p, X_train, X_train, params)
    align = kernel_alignment(K, y_train)
    return -align  # On maximise l'alignement


# Optimisation sur grille
scale_values = np.linspace(0.1, 3.0, 20)
alignments = []

for s in scale_values:
    align = -objective(s)
    alignments.append(align)

best_idx = np.argmax(alignments)
best_scale = scale_values[best_idx]
best_align = alignments[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(scale_values, alignments, 'o-', color='#8E44AD')
plt.axvline(x=best_scale, color='red', ls='--', alpha=0.5, label=f'Meilleur : scale={best_scale:.2f}')
plt.xlabel("Paramètre d'échelle")
plt.ylabel('Alignment du kernel')
plt.title('Quantum Kernel Alignment : optimisation du paramètre de scale')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Meilleur scale : {best_scale:.2f} (alignment = {best_align:.4f})")
print(f"Pire scale     : {scale_values[np.argmin(alignments)]:.2f} (alignment = {min(alignments):.4f})")

---
## Partie D : Implémentation Qiskit Machine Learning

Qiskit ML fournit `FidelityQuantumKernel` et `QSVC` pour une implémentation optimisée du QSVM.

In [ ]:
if QISKIT_ML_AVAILABLE:
    from qiskit.circuit.library import ZZFeatureMap
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    from qiskit_machine_learning.algorithms import QSVC
    from qiskit_aer import AerSimulator

    # Feature map ZZFeatureMap
    feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
    print("Feature map ZZFeatureMap créée")
    print(feature_map.decompose().draw())

    # Création du kernel quantique
    quantum_kernel = FidelityQuantumKernel(
        feature_map=feature_map,
        enable_clearing=True
    )

    # Matrice de kernel
    print("\nCalcul de la matrice de kernel avec Qiskit...")
    K_qiskit_train = quantum_kernel.evaluate(x_vec=X_train[:20])  # Sous-ensemble pour la vitesse

    # QSVC
    qsvc = QSVC(quantum_kernel=quantum_kernel)
    qsvc.fit(X_train, y_train)
    y_pred_qiskit = qsvc.predict(X_test)
    acc_qiskit = accuracy_score(y_test, y_pred_qiskit)

    print(f"\n=== Résultats QSVC (Qiskit ML) ===")
    print(f"Accuracy : {acc_qiskit:.4f}")
else:
    print("Qiskit Machine Learning n'est pas installé.")
    print("Installez-le avec : pip install qiskit-machine-learning")
    print("\nSimulation des résultats pour la démonstration :")
    # On utilise le kernel PennyLane comme approximation
    K_train = build_kernel_matrix(kernel_fn, X_train, X_train)
    svm_demo = SVC(kernel='precomputed').fit(K_train, y_train)
    K_test = build_kernel_matrix(kernel_fn, X_test, X_train)
    y_pred_demo = svm_demo.predict(K_test)
    acc_demo = accuracy_score(y_test, y_pred_demo)
    print(f"Accuracy (kernel PennyLane, SVC precomputed) : {acc_demo:.4f}")
    print(f"(Cette cellule utilise PennyLane en remplacement de Qiskit ML)")

---
## Partie E : Comparaison SVM classique vs. QSVM

Nous comparons le QSVM avec des SVM classiques utilisant différents noyaux : RBF, polynomial, et linéaire.

In [ ]:
# Recharger les données complètes
dataset_name = 'wine'
if dataset_name == 'wine':
    data = load_wine()
else:
    data = load_iris()

X, y = data.data, data.target

# Binariser (2 classes) pour comparaison
if len(np.unique(y)) > 2:
    mask = y != 2
    X, y = X[mask], y[mask]

print(f"Dataset : {dataset_name}, classes={np.unique(y)}, features={X.shape[1]}")

# Normalisation pour les SVM classiques
scaler_std = StandardScaler()
X_scaled = scaler_std.fit_transform(X)

# Normalisation [0, π] pour le kernel quantique
scaler_mm = MinMaxScaler(feature_range=(0, np.pi))
X_q = scaler_mm.fit_transform(X)

# Sélection des features (max 4 pour le kernel quantique)
n_features = min(4, X.shape[1])
X_scaled = X_scaled[:, :n_features]
X_q = X_q[:, :n_features]

X_train_s, X_test_s, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
X_train_q, X_test_q, _, _ = train_test_split(X_q, y, test_size=0.3, random_state=42)

# SVM classiques
classical_kernels = {
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'SVM (poly)': SVC(kernel='poly', C=1.0, degree=3),
    'SVM (linéaire)': SVC(kernel='linear', C=1.0),
}

results = {}
print("\n=== SVM classiques ===")
for name, model in classical_kernels.items():
    model.fit(X_train_s, y_train)
    acc = model.score(X_test_s, y_test)
    results[name] = acc
    print(f"  {name:20s} : accuracy = {acc:.4f}")

# QSVM (avec kernel PennyLane)
kernel_fn = create_fidelity_kernel(n_qubits=n_features, n_layers=1)
_, _, acc_qsvm = qsvm_pipeline(X_train_q, y_train, X_test_q, y_test, kernel_fn)
results['QSVM (fidelity kernel)'] = acc_qsvm
print(f"  {'QSVM (fidelity kernel)':20s} : accuracy = {acc_qsvm:.4f}")

# Comparaison visuelle
plt.figure(figsize=(8, 5))
colors_plot = ['#3498DB', '#2ECC71', '#F39C12', '#E74C3C']
bars = plt.bar(results.keys(), results.values(), color=colors_plot, alpha=0.8, edgecolor='black')
plt.ylabel('Accuracy')
plt.title(f'Comparaison SVM classiques vs. QSVM — {dataset_name}')
plt.ylim(0.5, 1.0)
for bar, acc in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Analyse :** Le QSVM atteint des performances compétitives avec les SVM classiques, surtout quand le nombre de features est faible ($\leq 4$). Le kernel RBF reste une baseline solide. L'avantage potentiel du QSVM réside dans sa capacité à accéder à un espace de Hilbert de très grande dimension, ce qui peut être bénéfique pour des données avec une structure complexe.

---
## Exercices

1. **Feature maps variées :** Testez différents encodages (`AngleEmbedding`, `IQPEmbedding`, `ZZFeatureMap`). Lequel donne le meilleur alignment ?

2. **Kernel alignment avancé :** Optimisez un paramètre par feature (vecteur $\vec{s}$ au lieu d'un scalaire). L'alignement s'améliore-t-il ?

3. **Dataset complet multiclasse :** Adaptez le QSVM pour la classification Iris à 3 classes (one-vs-one). Comparez avec SVM classique.

4. **Concentration du kernel :** Calculez la variance des éléments de la matrice de kernel pour $n=2$ à $n=10$ features. Observez-vous une concentration exponentielle (tous les $k_{ij} \to 1/2^n$) ?

5. **QSVC Qiskit ML :** Si Qiskit ML est installé, comparez le temps de calcul de `FidelityQuantumKernel` vs. votre implémentation PennyLane manuelle.

6. **(Avancé) Variational Quantum Kernel :** Implémentez un kernel variationnel où un circuit paramétré $V(\theta)$ est inséré entre l'encoding et l'adjoint : $k(x_i, x_j) = |\langle 0| V^\dagger(\theta) U^\dagger(x_j) U(x_i) V(\theta) |0\rangle|^2$.